# Group Chat Partner Launch Review (A2A)

| Field | Value |
| --- | --- |
| Scenario id | `group-chat-partner-launch-review` |
| Pattern | `group-chat` |
| API | `Invocations API` |

**Learning goal:** Learn how an invocation can convene a group chat whose partner seats are remote peer agents reached over the A2A protocol -- the orchestration is unchanged; only where two participants live changes.

> Use group chat over A2A when a decision needs seats from other organizations whose agents, models, and facts you do not host.


In [ ]:
import os
import re as _re

from IPython.display import HTML, display


_AGENT_COLORS = ("#3868c8", "#b0530f", "#2f7d4f", "#7d3f98", "#a3374b", "#0f7d8a", "#8a6d0f", "#54596b")

_APTOS_STYLE = """
<style>
:root { --jp-content-font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif; }
.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .rendered_html, .jp-OutputArea-output {
    font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif;
    line-height: 1.55;
}
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
    font-family: 'Aptos Display', 'Aptos', 'Segoe UI', sans-serif;
    font-weight: 600;
}
.maf-callout {
    border-left: 4px solid #3868c8; border-radius: 6px; padding: 0.6em 0.9em;
    margin: 0.6em 0; background: rgba(56, 104, 200, 0.08);
}
.maf-roster { display: flex; flex-wrap: wrap; gap: 0.6em; margin: 0.4em 0; }
.maf-card {
    border: 1px solid rgba(128, 128, 128, 0.35); border-radius: 8px;
    padding: 0.55em 0.8em; min-width: 14em; max-width: 24em; flex: 1;
}
.maf-card b { display: block; margin-bottom: 0.15em; }
.maf-card small { opacity: 0.75; }
.maf-chip {
    display: inline-block; border-radius: 999px; padding: 0 0.6em; margin: 0.2em 0.2em 0 0;
    font-size: 0.78em; border: 1px solid rgba(128, 128, 128, 0.4);
}
.maf-turn {
    border-left: 4px solid var(--maf-agent, #54596b); border-radius: 6px;
    padding: 0.45em 0.8em; margin: 0.45em 0; background: rgba(128, 128, 128, 0.07);
    white-space: pre-wrap;
}
.maf-turn b { color: var(--maf-agent, inherit); }
</style>
"""


def apply_notebook_style() -> str:
    display(HTML(_APTOS_STYLE))
    return _APTOS_STYLE


def _escape_html(value) -> str:
    import html as _html

    return _html.escape(str(value))


def agent_color(name: str) -> str:
    """Deterministic per-agent accent color, stable across cells and runs."""

    return _AGENT_COLORS[sum(ord(ch) for ch in name) % len(_AGENT_COLORS)]


def render_callout(text: str) -> None:
    display(HTML("<div class='maf-callout'>" + _escape_html(text) + "</div>"))


def render_roster(scenario) -> None:
    """Render the agent roster as color-accented cards with tool chips."""

    cards = []
    for spec in scenario.agents:
        chips = "".join(
            "<span class='maf-chip'>" + _escape_html(tool) + "</span>" for tool in spec.mcp_tools
        ) or "<span class='maf-chip'>instructions only</span>"
        cards.append(
            "<div class='maf-card' style='border-top: 3px solid " + agent_color(spec.name) + "'>"
            + "<b>" + _escape_html(spec.name) + "</b>"
            + "<small>" + _escape_html(spec.description) + "</small>"
            + "<div>" + chips + "</div></div>"
        )
    display(HTML("<div class='maf-roster'>" + "".join(cards) + "</div>"))


_TURN_LABEL = _re.compile(r"^\[([A-Za-z0-9_]+)\]\s*", _re.MULTILINE)


def render_transcript(text: str) -> None:
    """Render workflow output as color-coded per-agent turns; plain print fallback."""

    pieces = _TURN_LABEL.split(text)
    turns = []
    preamble = pieces[0].strip()
    if preamble:
        turns.append("<div class='maf-turn'>" + _escape_html(preamble) + "</div>")
    for label, body in zip(pieces[1::2], pieces[2::2]):
        turns.append(
            "<div class='maf-turn' style='--maf-agent: " + agent_color(label) + "'>"
            + "<b>" + _escape_html(label) + "</b><br>" + _escape_html(body.strip()) + "</div>"
        )
    if turns:
        display(HTML("<div>" + "".join(turns) + "</div>"))
    else:
        print(text)


OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:14b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MCP_TOOL_FUNCTIONS: dict[str, object] = {}

apply_notebook_style()
print(f"Ollama target: {OLLAMA_HOST} / {OLLAMA_MODEL}")


## Pattern: Group Chat

Group chat orchestration creates a visible multi-agent discussion. A selector function chooses the next participant round-robin, and a per-scenario termination function checks the closing message of each full cycle, so the synthesizer or chair always speaks last.

Best fit: decisions that benefit from critique, tradeoffs, and a short transcript.

## API Shape

**Invocations API -- per-request job payload shape.** Each request body carries `scenario`, `pattern`, `task`, `artifacts`, and `constraints`. The caller controls which orchestration runs per invocation. This fits webhooks, CI pipelines, schedulers, and service-to-service calls where the task definition changes with every request.

This scenario seats remote partner agents in the council over the A2A protocol. MCP (scenarios 11-16) connected agents to tools; A2A connects agents to peer agents owned by other organizations. The orchestration below is the same group chat used elsewhere -- only where two participants live changes.

## Pattern Anatomy

| Dimension | Detail |
| --- | --- |
| Control flow | Agents take turns in cycles; the last agent closes each cycle and can end the chat. |
| Coordination | A selector function rotates speakers; termination is checked only at cycle boundaries. |
| Output | The transcript shows critique, refinement, and a closing verdict. |
| Best when | Use when visible debate improves the answer. |

## Instruction-Led LLM Agents

| Agent | Role | Capabilities |
| --- | --- | --- |
| `ProductLeadAgent` | Argues product readiness and launch scope. | Instructions only |
| `OperationsLeadAgent` | Argues support and operational readiness. | Instructions only |
| `PartnerSolutionsAgent` | ISV partner seat, reached over A2A. | Instructions only |
| `ExternalComplianceAgent` | External audit firm seat, reached over A2A. | Instructions only |
| `JointLaunchChairAgent` | Synthesizes the joint council and closes each round. | Instructions only |

> **Instructor note:** Each row is an LLM-backed agent with role instructions.
> Most agents rely on instructions alone; enterprise and quote-to-cash agents may
> also call domain tools for grounded context.


## A2A Partner Context

Two council seats belong to *partner organizations* and are reached over the
**A2A (Agent2Agent) protocol**. Where MCP connects an agent to tools, A2A connects an
agent to *peer agents*: each partner publishes an agent card over HTTP and answers
JSON-RPC messages; its model, instructions, and facts stay behind its own boundary.
In production those agents run in the partner's infrastructure; this notebook hosts
deterministic stand-ins in-process so every cell runs without credentials or a second
terminal. The next cells walk the protocol on-ramp one step at a time: partner facts,
partner behavior, hosting, agent-card discovery, and a direct client round-trip --
all before any orchestration exists.


In [ ]:
PARTNER_FIXTURES = {
    "partner-solutions": {
        "organization": "Fabrikam Integrations (ISV partner)",
        "integration_certification_expires": "2026-07-20",
        "launch_window": "2026-07-15 to 2026-07-31",
        "nightly_integration_tests": "47 passing, 1 failing (bulk-export, since Tuesday)",
        "connector_version": "2.3.1",
        "notes": "Certification expires mid launch window; the renewal audit is booked for 2026-07-18.",
    },
    "compliance": {
        "organization": "Meridian Assurance (external audit firm)",
        "soc2_status": "current",
        "joint_data_processing_addendum": "signed",
        "open_findings": 1,
        "open_finding_detail": "Partner telemetry retention is 120 days; the joint standard requires 90.",
        "notes": "The open finding needs a remediation date before joint go-live.",
    },
}

PARTNER_SEATS = {
    "partner-solutions": ("PartnerSolutionsAgent", "ISV partner agent: argues partner-side integration readiness."),
    "compliance": ("ExternalComplianceAgent", "External audit firm agent: argues certification and compliance status."),
}


def partner_reply(path: str, question: str | None = None) -> str:
    """The fixture-grounded answer a partner agent gives -- zero LLM calls.

    Question-aware but deterministic: fact keys whose words overlap the
    question are returned (plus notes); the full sheet is the fallback.
    """

    import re as _re

    facts = PARTNER_FIXTURES[path]
    name, _ = PARTNER_SEATS[path]
    selected = {key: value for key, value in facts.items() if key != "organization"}
    if question:
        words = {word for word in _re.findall(r"[a-z0-9]+", question.lower()) if len(word) > 3}
        matched = {key: value for key, value in selected.items() if set(key.split("_")) & words}
        if matched:
            matched.setdefault("notes", facts["notes"])
            selected = matched
    lines = [f"{name} ({facts['organization']}) reports:"]
    for key, value in selected.items():
        lines.append(f"- {key.replace('_', ' ')}: {value}")
    return "\n".join(lines)


# Demo (offline): the partner behavior is a function over its facts, and it
# answers the question it was asked -- compare the full sheet vs a targeted ask.
print(partner_reply("partner-solutions"))
print()
print(partner_reply("partner-solutions", "When does the integration certification expire?"))


In [ ]:
import socket
import threading
import time

import uvicorn
from starlette.applications import Starlette

from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill

from agent_framework import AgentResponse, BaseAgent, Message
from agent_framework.a2a import A2AExecutor


class DeterministicPartnerAgent(BaseAgent):
    """The agent behind the A2A endpoint: answers from PARTNER_FIXTURES."""

    def __init__(self, path: str, **kwargs) -> None:
        super().__init__(**kwargs)
        self._path = path

    async def run(self, messages=None, *, session=None, **kwargs):
        question = messages if isinstance(messages, str) else getattr(messages, "text", "") or ""
        if isinstance(messages, (list, tuple)):
            question = " ".join(getattr(m, "text", "") or str(m) for m in messages)
        reply = partner_reply(self._path, question)
        return AgentResponse(messages=[Message(role="assistant", contents=[reply])])

    async def run_stream(self, messages=None, *, session=None, **kwargs):
        yield await self.run(messages, session=session, **kwargs)


def _partner_routes(path: str, base_url: str) -> list:
    name, description = PARTNER_SEATS[path]
    card = AgentCard(
        name=name,
        description=description,
        version="1.0.0",
        supported_interfaces=[AgentInterface(url=f"{base_url}/{path}", protocol_binding="JSONRPC")],
        capabilities=AgentCapabilities(streaming=False),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[AgentSkill(id=f"{path}-launch-review", name="Joint launch review", description=description, tags=["a2a"])],
    )
    executor = A2AExecutor(DeterministicPartnerAgent(path, name=name, description=description))
    handler = DefaultRequestHandler(agent_executor=executor, task_store=InMemoryTaskStore(), agent_card=card)
    # Flat prefixed routes: the JSON-RPC endpoint lives at exactly /<path>.
    return create_agent_card_routes(
        card, card_url=f"/{path}/.well-known/agent-card.json"
    ) + create_jsonrpc_routes(handler, rpc_url=f"/{path}")


with socket.socket() as _sock:
    _sock.bind(("127.0.0.1", 0))
    A2A_PORT = _sock.getsockname()[1]
A2A_BASE_URL = f"http://127.0.0.1:{A2A_PORT}"

_routes = []
for _path in PARTNER_SEATS:
    _routes.extend(_partner_routes(_path, A2A_BASE_URL))
_app = Starlette(routes=_routes)
_uvicorn_server = uvicorn.Server(uvicorn.Config(_app, host="127.0.0.1", port=A2A_PORT, log_level="error"))
threading.Thread(target=_uvicorn_server.run, daemon=True).start()
_deadline = time.time() + 10
while not _uvicorn_server.started:
    if time.time() > _deadline:
        raise RuntimeError("Partner A2A server did not start.")
    time.sleep(0.05)

os.environ["A2A_PARTNER_BASE_URL"] = A2A_BASE_URL
print(f"Partner A2A server up: {A2A_BASE_URL}  (seats: " + ", ".join(n for n, _ in PARTNER_SEATS.values()) + ")")


In [ ]:
import httpx

# Demo (offline): protocol discovery -- fetch each partner's agent card over HTTP.
for _path, (_name, _desc) in PARTNER_SEATS.items():
    _card = httpx.get(f"{A2A_BASE_URL}/{_path}/.well-known/agent-card.json", timeout=5).json()
    _iface = (_card.get("supportedInterfaces") or [{}])[0]
    print(f"{_card.get('name')}: {_iface.get('url')} [{_iface.get('protocolBinding', 'JSONRPC')}]")
    print(f"  {_card.get('description')}")


In [ ]:
from agent_framework.a2a import A2AAgent

# Demo (offline): one direct A2A round-trip before any orchestration exists.
_partner = A2AAgent(name="PartnerSolutionsAgent", url=f"{A2A_BASE_URL}/partner-solutions")
_reply = await _partner.run("Report partner-side launch readiness for the July window.")
render_transcript("[PartnerSolutionsAgent] " + (_reply.text or ""))


In [ ]:
import json
from dataclasses import dataclass
from typing import Any, Sequence


@dataclass(frozen=True)
class AgentSpec:
    name: str
    description: str
    instructions: str
    mcp_tools: tuple[str, ...] = ()
    mcp_server: str = "enterprise_context"
    route_keywords: tuple[str, ...] = ()
    a2a_url: str | None = None


@dataclass(frozen=True)
class ScenarioSpec:
    id: str
    pattern: str
    title: str
    learning_goal: str
    when_to_use: str
    sample_task: str
    agents: tuple[AgentSpec, ...]
    handoff_finisher: str | None = None
    concurrent_synthesizer: str | None = None
    termination_phrases: tuple[str, ...] = ()


SCENARIO_DATA = json.loads(
    r"""
{
  "id": "group-chat-partner-launch-review",
  "pattern": "group-chat",
  "title": "Group Chat Partner Launch Review (A2A)",
  "learning_goal": "Learn how an invocation can convene a group chat whose partner seats are remote peer agents reached over the A2A protocol -- the orchestration is unchanged; only where two participants live changes.",
  "when_to_use": "Use group chat over A2A when a decision needs seats from other organizations whose agents, models, and facts you do not host.",
  "sample_task": "Run a joint launch review job for the co-sold analytics integration (window 2026-07-15 to 2026-07-31). Partner-side integration status and external compliance status are owned by partner organizations -- their agents must speak for those facts.",
  "handoff_finisher": null,
  "concurrent_synthesizer": null,
  "termination_phrases": [
    "final recommendation"
  ],
  "agents": [
    {
      "name": "ProductLeadAgent",
      "description": "Argues product readiness and launch scope.",
      "instructions": "Argue product readiness for the co-sold analytics integration: scope clarity, launch goals, and customer commitments inside the stated window.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "OperationsLeadAgent",
      "description": "Argues support and operational readiness.",
      "instructions": "Argue operational readiness: support coverage, rollback, monitoring, and incident ownership across two organizations.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    },
    {
      "name": "PartnerSolutionsAgent",
      "description": "ISV partner seat, reached over A2A.",
      "instructions": "Remote partner agent: its instructions and facts live with the partner organization and are served behind its A2A agent card.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": "/partner-solutions"
    },
    {
      "name": "ExternalComplianceAgent",
      "description": "External audit firm seat, reached over A2A.",
      "instructions": "Remote audit-firm agent: its instructions and facts live with the audit firm and are served behind its A2A agent card.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": "/compliance"
    },
    {
      "name": "JointLaunchChairAgent",
      "description": "Synthesizes the joint council and closes each round.",
      "instructions": "Synthesize the local and partner positions each round, calling out conflicts between the launch window and partner-side facts. When the council has converged, end your turn with a line 'FINAL RECOMMENDATION: <launch or hold> - <one-sentence rationale>'.",
      "mcp_tools": [],
      "mcp_server": "enterprise_context",
      "route_keywords": [],
      "a2a_url": null
    }
  ]
}
    """
)
AGENTS = tuple(
    AgentSpec(
        name=item["name"],
        description=item["description"],
        instructions=item["instructions"],
        mcp_tools=tuple(item.get("mcp_tools", [])),
        mcp_server=item.get("mcp_server", "enterprise_context"),
        route_keywords=tuple(item.get("route_keywords", [])),
        a2a_url=item.get("a2a_url"),
    )
    for item in SCENARIO_DATA["agents"]
)
SCENARIO = ScenarioSpec(
    id=SCENARIO_DATA["id"],
    pattern=SCENARIO_DATA["pattern"],
    title=SCENARIO_DATA["title"],
    learning_goal=SCENARIO_DATA["learning_goal"],
    when_to_use=SCENARIO_DATA["when_to_use"],
    sample_task=SCENARIO_DATA["sample_task"],
    agents=AGENTS,
    handoff_finisher=SCENARIO_DATA.get("handoff_finisher"),
    concurrent_synthesizer=SCENARIO_DATA.get("concurrent_synthesizer"),
    termination_phrases=tuple(SCENARIO_DATA.get("termination_phrases", [])),
)


def tools_for_agent(spec: AgentSpec) -> list[object]:
    tools: list[object] = []
    for tool_name in spec.mcp_tools:
        try:
            tools.append(MCP_TOOL_FUNCTIONS[tool_name])
        except KeyError as exc:
            raise ValueError(f"Missing inlined tool '{tool_name}' for {spec.name}.") from exc
    return tools


def scenario_summary(scenario: ScenarioSpec) -> dict[str, str]:
    return {
        "id": scenario.id,
        "title": scenario.title,
        "pattern": scenario.pattern,
        "learning_goal": scenario.learning_goal,
        "when_to_use": scenario.when_to_use,
        "sample": getattr(scenario, "sample_task"),
    }


def agent_capability_map(scenario: ScenarioSpec) -> list[dict[str, Any]]:
    return [
        {
            "agent": spec.name,
            "description": spec.description,
            "instructions": spec.instructions,
            "mcp_tools": list(spec.mcp_tools),
            "mcp_server": spec.mcp_server if spec.mcp_tools else None,
        }
        for spec in scenario.agents
    ]


def mcp_tool_context(scenario: ScenarioSpec) -> dict[str, Any]:
    tools_by_agent = {spec.name: list(spec.mcp_tools) for spec in scenario.agents if spec.mcp_tools}
    all_tools_used = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    return {
        "uses_mcp": bool(all_tools_used),
        "tools_by_agent": tools_by_agent,
        "all_tools_used": all_tools_used,
    }
def build_invocation_prompt(payload: dict[str, object]) -> str:
    artifacts = "\n".join(f"- {item}" for item in payload.get("artifacts", [])) or "- No artifacts supplied."
    constraints = "\n".join(f"- {item}" for item in payload.get("constraints", [])) or "- No explicit constraints."
    return (
        f"Scenario: {payload['scenario']} - {SCENARIO.title}\n"
        f"Pattern: {payload['pattern']}\n"
        f"Learning goal: {SCENARIO.learning_goal}\n"
        f"Subject: {payload['subject']}\n"
        f"Task: {payload['task']}\n\n"
        f"Artifacts:\n{artifacts}\n\n"
        f"Constraints:\n{constraints}\n\n"
        "Session context:\nNo prior turns for this session.\n\n"
        "Return actionable findings. Do not claim to have inspected artifacts beyond the provided names and context."
    )

MAX_TOKENS = 500 if SCENARIO.pattern == "magentic" else 250
INVOCATION_PAYLOAD = {
    "scenario": SCENARIO.id,
    "pattern": SCENARIO.pattern,
    "task": SCENARIO.sample_task,
    "subject": "notebook sample",
    "artifacts": [],
    "constraints": [],
    "stream": False,
}
SAMPLE_PROMPT = build_invocation_prompt(INVOCATION_PAYLOAD)

render_roster(SCENARIO)
print(json.dumps(scenario_summary(SCENARIO), indent=2))
print(json.dumps(agent_capability_map(SCENARIO), indent=2))
if mcp_tool_context(SCENARIO)["uses_mcp"]:
    print(json.dumps(mcp_tool_context(SCENARIO), indent=2))


In [ ]:
from dataclasses import dataclass
from typing import Any

from agent_framework.ollama import OllamaChatClient


DEFAULT_OLLAMA_TEMPERATURE = 0.0
DEFAULT_OLLAMA_NUM_CTX = 8192
DEFAULT_OLLAMA_KEEP_ALIVE = "10m"
DEFAULT_OLLAMA_THINK = False

_UNSUPPORTED_OLLAMA_CHAT_OPTIONS = {
    "allow_multiple_tool_calls",
    "conversation_id",
    "logit_bias",
    "metadata",
    "store",
    "user",
}


@dataclass(frozen=True)
class OllamaAgentConfig:
    model: str
    host: str
    temperature: float
    num_ctx: int
    max_tokens: int
    keep_alive: str
    think: bool

    def default_options(self) -> dict[str, Any]:
        return {
            "temperature": self.temperature,
            "num_ctx": self.num_ctx,
            "max_tokens": self.max_tokens,
            "keep_alive": self.keep_alive,
            "think": self.think,
        }


def parse_env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true or false.")


def build_ollama_config(
    *,
    model: str | None = None,
    host: str | None = None,
    temperature: float | None = None,
    num_ctx: int | None = None,
    max_tokens: int | None = None,
    keep_alive: str | None = None,
    think: bool | None = None,
) -> OllamaAgentConfig:
    return OllamaAgentConfig(
        model=model or os.getenv("OLLAMA_MODEL") or "qwen3:14b",
        host=host or os.getenv("OLLAMA_HOST") or "http://localhost:11434",
        temperature=temperature
        if temperature is not None
        else float(os.getenv("OLLAMA_TEMPERATURE", str(DEFAULT_OLLAMA_TEMPERATURE))),
        num_ctx=num_ctx if num_ctx is not None else int(os.getenv("OLLAMA_NUM_CTX", str(DEFAULT_OLLAMA_NUM_CTX))),
        max_tokens=max_tokens if max_tokens is not None else int(os.getenv("OLLAMA_MAX_TOKENS", "500")),
        keep_alive=keep_alive or os.getenv("OLLAMA_KEEP_ALIVE") or DEFAULT_OLLAMA_KEEP_ALIVE,
        think=think if think is not None else parse_env_bool("OLLAMA_THINK", DEFAULT_OLLAMA_THINK),
    )


class ScenarioOllamaChatClient(OllamaChatClient):
    def _prepare_options(self, messages: Any, options: Any) -> dict[str, Any]:
        filtered = {
            key: value
            for key, value in dict(options).items()
            if key not in _UNSUPPORTED_OLLAMA_CHAT_OPTIONS
        }
        return super()._prepare_options(messages, filtered)


def make_agent(spec: Any, *, config: OllamaAgentConfig | None = None) -> Any:
    if spec.a2a_url:
        from agent_framework.a2a import A2AAgent

        url = spec.a2a_url
        if not url.startswith("http"):
            url = os.getenv("A2A_PARTNER_BASE_URL", "http://localhost:8765").rstrip("/") + url
        return A2AAgent(name=spec.name, description=spec.description, url=url)

    resolved = config or build_ollama_config()
    instructions = f"You are {spec.name}. {spec.instructions}"
    tools = tools_for_agent(spec)
    return ScenarioOllamaChatClient(host=resolved.host, model=resolved.model).as_agent(
        name=spec.name,
        description=spec.description,
        instructions=instructions,
        tools=tools or None,
        default_options=resolved.default_options(),
        require_per_service_call_history_persistence=True,
    )


print("Agent factory ready: make_agent(spec) creates an instruction-led Ollama agent "
      "with its granted tools attached.")


In [ ]:
import re
from typing import Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)


_TRANSCRIPT_KEY = "transcript"
_STOPWORDS = {"agent", "specialist", "the", "and", "for", "with", "that", "from", "into"}


def make_request(text: str) -> AgentExecutorRequest:
    return AgentExecutorRequest(messages=[Message(role="user", contents=[text])])


def response_text(response: AgentExecutorResponse) -> str:
    agent_response = getattr(response, "agent_response", None)
    return (getattr(agent_response, "text", None) or "").strip()


def _append_transcript(ctx: WorkflowContext[Any], author: str, text: str) -> list[str]:
    transcript = list(ctx.get_state(_TRANSCRIPT_KEY) or [])
    transcript.append(f"[{author}] {text}")
    ctx.set_state(_TRANSCRIPT_KEY, transcript)
    return transcript


class PromptDispatchExecutor(Executor):
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        ctx.set_state("prompt", prompt)
        ctx.set_state(_TRANSCRIPT_KEY, [])
        await ctx.send_message(make_request(prompt))


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _agents_for(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> list[Any]:
    return [make_agent(spec, config=config) for spec in scenario.agents]


def _agent_executor(spec_index: int, scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> AgentExecutor:
    spec = scenario.agents[spec_index]
    return AgentExecutor(make_agent(spec, config=config), id=_slug(spec.name))



print("Workflow plumbing ready: dispatch executor, shared transcript state, and "
      "request/response helpers.")


In [ ]:
def make_group_chat_termination(phrases: tuple[str, ...], participant_count: int, max_cycles: int = 2) -> Any:
    def should_stop(messages: list[Any]) -> bool:
        assistant = [m for m in messages if getattr(m, "role", None) == "assistant"]
        if not assistant or len(assistant) % participant_count != 0:
            return False
        if len(assistant) >= max_cycles * participant_count:
            return True
        last_text = (getattr(assistant[-1], "text", "") or "").lower()
        return bool(phrases) and all(phrase in last_text for phrase in phrases)

    return should_stop



# Demo (offline): termination only fires when the closing agent ends a full cycle.
class _DemoMsg:
    def __init__(self, text: str) -> None:
        self.role = "assistant"
        self.text = text


_n = len(SCENARIO.agents)
_phrase = " ".join(SCENARIO.termination_phrases) or "final recommendation"
_stop = make_group_chat_termination(SCENARIO.termination_phrases, _n)
print("mid-cycle, phrase present  ->", _stop([_DemoMsg(_phrase)] * max(1, _n - 1)))
print("cycle end, no phrase       ->", _stop([_DemoMsg("still debating")] * _n))
print("cycle end, phrase present  ->", _stop([_DemoMsg("x")] * (_n - 1) + [_DemoMsg(_phrase)]))
print("after two full cycles      ->", _stop([_DemoMsg("x")] * (2 * _n)))


In [ ]:
def build_group_chat_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    from agent_framework.orchestrations import GroupChatBuilder

    participants = _agents_for(scenario, config=config)

    def round_robin_selector(state: Any) -> str:
        participant_names = list(state.participants.keys())
        return participant_names[state.current_round % len(participant_names)]

    return GroupChatBuilder(
        participants=participants,
        selection_func=round_robin_selector,
        termination_condition=make_group_chat_termination(
            scenario.termination_phrases, len(scenario.agents)
        ),
        intermediate_output_from=participants,
    ).build()



def build_workflow(
    scenario: ScenarioSpec = SCENARIO,
    *,
    max_tokens: int | None = None,
    **config_overrides: Any,
) -> Any:
    config = build_ollama_config(max_tokens=max_tokens or MAX_TOKENS, **config_overrides)
    return build_group_chat_workflow(scenario, config=config)


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
print(
    f"Built the {SCENARIO.pattern} workflow over {len(SCENARIO.agents)} agents: "
    + type(workflow).__name__
)


## Flow Diagram

The diagram below shows 5 participants in a round-robin loop with a code-defined termination function attached to the Invocations API /invocations.
Solid arrows are orchestration edges. Dashed arrows (`-.->`) are tool calls.
Domain tool nodes use a stadium shape.


In [ ]:
import base64
import html
from dataclasses import dataclass

from IPython.display import HTML, display


@dataclass(frozen=True)
class ScenarioFlowDiagram:
    title: str
    mermaid: str
    image_url: str


def scenario_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _group_chat_diagram(scenario, api_boundary="Invocations API /invocations", input_label="Job payload")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_scenario_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = scenario_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram



def _group_chat_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append("    orchestrator --> selector{Round-robin selector}")
    previous = "selector"
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    {previous} --> {node}[{_label(agent.name)}]")
        previous = node
        pairs.append((agent, node))
    lines.append(f"    {previous} --> stop{{Termination condition}}")
    lines.append("    stop -->|continue| selector")
    lines.append("    stop -->|done| output[Invocation summary]")
    remote_nodes = [node for agent, node in pairs if getattr(agent, "a2a_url", None)]
    if remote_nodes:
        lines.append("    subgraph partner_org[Partner organizations via A2A]")
        for node in remote_nodes:
            lines.append(f"        {node}")
        lines.append("    end")
        for node in remote_nodes:
            lines.append(f"    {node} -.->|A2A JSON-RPC| a2a_card([agent card])")
    lines.extend(_mcp_tool_links(pairs))
    return "\n".join(lines)



def _header(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> list[str]:
    return [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label(input_label)}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
    ]


def _mcp_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in agent.mcp_tools:
            lines.append(f"    {node_id} -.->|mcp tool| tool_{tool}([{_label(tool)}])")
    return lines


def quote_to_cash_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _quote_to_cash_source(scenario, api_boundary="Invocations API /invocations")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Quote-To-Cash Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_quote_to_cash_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = quote_to_cash_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _quote_to_cash_source(scenario: ScenarioSpec, *, api_boundary: str) -> str:
    names = {agent.name for agent in scenario.agents}

    def node(role: str) -> str:
        return role if role in names else next(iter(names))

    lines = [
        "%%{init: {'theme': 'neutral'}}%%",
    "flowchart TD",
        f"    client[{_label('Quote request begins in CRM')}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
        f"    orchestrator --> crm[{_label('CRM system')}]",
        f"    crm -->|wave 1| trigger[{_label(node('QuoteTriggerAgent'))}]",
        f"    crm -->|wave 1| customer[{_label(node('CustomerContextAgent'))}]",
        f"    trigger --> store1[({_label('Orchestration store: customer context')})]",
        "    customer --> store1",
        f"    store1 -. {_label('deallocate wave 1')} .-> freed1(({_label('agents released')}))",
        f"    store1 --> product[{_label('Product / SKU system')}]",
        f"    product -->|wave 2| sku[{_label(node('SkuDiscoveryAgent'))}]",
        f"    product -->|wave 2| fit[{_label(node('ProductFitAgent'))}]",
        f"    sku --> store2[({_label('Orchestration store: product context')})]",
        "    fit --> store2",
        f"    store2 -. {_label('deallocate wave 2')} .-> freed2(({_label('agents released')}))",
        f"    store2 --> pricingsys[{_label('Pricing / finance / legal system')}]",
        f"    pricingsys -->|wave 3| pricing[{_label(node('PricingTermsAgent'))}]",
        f"    pricing --> generation[{_label(node('QuoteGenerationAgent'))}]",
        f"    generation --> quote[/{_label('Final quote package')}/]",
    ]
    return "\n".join(lines)


def _label(value: str) -> str:
    return value.replace('"', "'")


def _mermaid_image_url(mermaid: str) -> str:
    encoded = base64.urlsafe_b64encode(mermaid.encode("utf-8")).decode("ascii").rstrip("=")
    return f"https://mermaid.ink/img/{encoded}"


flow_diagram = display_scenario_flow(SCENARIO)
print(flow_diagram.mermaid)


In [ ]:
from collections.abc import Mapping, Sequence


def workflow_result_to_text(result: Any) -> str:
    outputs = result.get_outputs() if hasattr(result, "get_outputs") else result
    intermediate = result.get_intermediate_outputs() if hasattr(result, "get_intermediate_outputs") else []
    if not outputs:
        intermediate_text = join_readable_outputs(intermediate)
        return clean_workflow_text(intermediate_text) or "No workflow output was produced."
    output_text = join_readable_outputs(outputs)
    if intermediate and should_use_intermediate_outputs(output_text):
        intermediate_text = join_readable_outputs(intermediate)
        if intermediate_text:
            return clean_workflow_text(intermediate_text)
    return clean_workflow_text(output_text) or "No readable workflow text was produced."


def join_readable_outputs(outputs: Any) -> str:
    return "\n\n".join(text for output in outputs if (text := agent_response_to_text(output)))


def should_use_intermediate_outputs(output_text: str) -> bool:
    normalized = output_text.strip().lower()
    if not normalized:
        return True
    if len(normalized) >= 160:
        return False
    markers = (
        "termination condition",
        "maximum reset count",
        "maximum stall count",
        "workflow terminated",
        "group chat has reached its termination condition",
    )
    return any(marker in normalized for marker in markers)


def agent_response_to_text(value: Any) -> str:
    text = clean_workflow_text(extract_text(value))
    return text


def clean_workflow_text(text: str) -> str:
    """Remove leading framework status lines when useful scenario text follows."""

    lines = text.splitlines()
    while lines and is_framework_status_line(lines[0]) and any(line.strip() for line in lines[1:]):
        lines.pop(0)
        while lines and not lines[0].strip():
            lines.pop(0)
    return "\n".join(lines).strip()


def is_framework_status_line(line: str) -> bool:
    normalized = line.strip().lower()
    return (
        normalized.startswith("invalid next speaker:")
        or normalized.startswith("magentic orchestrator:")
        or normalized.startswith("maximum consecutive function call errors reached")
    )


def extract_text(value: Any, seen: set[int] | None = None) -> str:
    if value is None:
        return ""
    if seen is None:
        seen = set()
    value_id = id(value)
    if value_id in seen:
        return ""
    seen.add(value_id)
    if isinstance(value, str):
        return "" if is_object_repr(value) else value
    text = getattr(value, "text", None)
    if isinstance(text, str) and text and not is_object_repr(text):
        return text
    messages = getattr(value, "messages", None)
    if messages:
        parts: list[str] = []
        for message in messages:
            author = getattr(message, "author_name", None) or getattr(message, "role", None) or "assistant"
            message_text = extract_text(message, seen)
            if message_text:
                parts.append(f"[{author}] {message_text}")
        if parts:
            return "\n".join(parts)
    contents = getattr(value, "contents", None)
    if contents:
        return "\n".join(part for content in contents if (part := extract_text(content, seen)))
    items = getattr(value, "items", None)
    if items and not callable(items):
        return "\n".join(part for item in items if (part := extract_text(item, seen)))
    result = getattr(value, "result", None)
    if result is not None:
        return extract_text(result, seen)
    if isinstance(value, Mapping):
        parts = [extract_text(value.get(key), seen) for key in ("text", "content", "message", "summary", "result")]
        return "\n".join(part for part in parts if part)
    if isinstance(value, Sequence) and not isinstance(value, (bytes, bytearray, str)):
        return "\n".join(part for item in value if (part := extract_text(item, seen)))
    fallback = str(value)
    return "" if is_object_repr(fallback) else fallback


def is_object_repr(value: str) -> bool:
    return value.startswith("<") and " object at 0x" in value and value.endswith(">")



def group_chat_learning_summary(
    scenario: ScenarioSpec,
    prompt: str,
    framework_text: str,
) -> str:
    """Explain a completed group-chat run when this framework build hides the transcript."""

    lines = [
        "Group chat completed.",
        "",
        f"Framework result: {framework_text.strip()}",
        "",
        "Learning view:",
        "- The workflow used Microsoft Agent Framework's GroupChatBuilder with LLM-backed participants.",
        "- Selection is code-defined round robin; termination is code-defined from assistant messages.",
        f"- The submitted scenario prompt was: {prompt}",
        "- Participant order:",
    ]
    for index, spec in enumerate(scenario.agents, start=1):
        tools = ", ".join(spec.mcp_tools) if spec.mcp_tools else "no domain tools"
        lines.append(f"  {index}. {spec.name}: {spec.description} ({tools})")
    tool_names = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    if tool_names:
        lines.append("- Grounding sources available to tool-enabled agents:")
        for tool_name in tool_names:
            lines.append(f"  - {tool_name}")
    lines.extend(
        [
            "",
            "Note: this local Agent Framework build returned the group-chat termination marker",
            "without exposing participant turns through get_intermediate_outputs(). The notebook",
            "keeps the framework run intact and prints this learning summary so the scenario",
            "still explains the orchestration shape and agent responsibilities.",
        ]
    )
    return "\n".join(lines)


print("Result formatting ready: workflow_result_to_text(...) turns framework events "
      "into readable text.")


## Live Run

Participants speak in round-robin order, and termination is only checked when the last agent closes a full cycle -- so the synthesizer always gets the final word. The chat ends early when the scenario's termination phrases appear in that closing message, and unconditionally after two full cycles. Intermediate outputs from each participant are surfaced alongside the final transcript.

> **Instructor note:** `qwen3:14b` runs with `think: False` by default (extended reasoning off).
> Set `OLLAMA_THINK=true` before the environment cell to enable chain-of-thought reasoning --
> useful when debugging unexpected routing decisions or tool call sequences.


In [ ]:
import io
from contextlib import redirect_stderr, redirect_stdout


workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
_framework_logs = io.StringIO()
with redirect_stdout(_framework_logs), redirect_stderr(_framework_logs):
    result = await workflow.run(SAMPLE_PROMPT)
framework_logs = _framework_logs.getvalue()
output_text = workflow_result_to_text(result)
if SCENARIO.pattern == "group-chat" and should_use_intermediate_outputs(output_text):
    output_text = group_chat_learning_summary(SCENARIO, SAMPLE_PROMPT, output_text)

if not output_text.strip():
    raise RuntimeError("Workflow completed but produced no readable text.")

render_transcript(output_text)


## What to Inspect

Read the transcript chronologically. Later turns should respond to earlier critiques rather than restarting the discussion. Termination is checked only at the end of each full cycle: the chat stops early when the scenario's termination phrases appear in the closing agent's message, or after two full cycles -- check which condition fired and why.

> **Scenario spotlight:** The partner certification expires mid launch window and one compliance finding is open -- both facts live only behind the A2A seats, so the chair's verdict must cite what the remote agents reported.


## Experiments

- Edit PARTNER_FIXTURES so the certification renews before the window opens, rerun the server cell onward, and compare the FINAL RECOMMENDATION.
- Change `INVOCATION_PAYLOAD['task']`, `subject`, `artifacts`, or `constraints` and rerun the live cell.
- Override `OLLAMA_MODEL` or `OLLAMA_HOST` before the environment cell to target a different local Ollama setup.
- Inspect `agent_capability_map(SCENARIO)` and tighten one agent's instructions to see how orchestration behavior changes.
- Lower `MAX_TOKENS` for faster runs or raise it when group-chat needs more room.
